In [1]:
import pandas as pd


In [2]:
prices = pd.read_csv(
    "../data/processed/se3_hourly_prices.csv", 
    parse_dates=["timestamp"]
)

weather = pd.read_csv(
    "../data/processed/se3_weather_forecast.csv",
    parse_dates=["timestamp"]
)

load = pd.read_csv(
    "../data/processed/se3_load_forecast.csv",
    parse_dates=["timestamp"]
)

In [3]:
print("Prices:", prices.shape)
print("Weather:", weather.shape)
print("Load:", load.shape)

Prices: (18816, 4)
Weather: (18816, 7)
Load: (18816, 4)


In [4]:
print(prices.columns)
print(weather.columns)
print(load.columns)

Index(['timestamp', 'price', 'count', 'valid'], dtype='str')
Index(['timestamp', 'temp_stockholm', 'wind_stockholm', 'temp_gothenburg',
       'wind_gothenburg', 'temp_orebro', 'wind_orebro'],
      dtype='str')
Index(['timestamp', 'load_forecast_mw', 'count', 'valid'], dtype='str')


In [5]:
prices = prices.rename(columns={
    "count": "price_count",
    "valid": "price_valid"
})

load = load.rename(columns={
    "count": "load_count",
    "valid": "load_valid"
})

In [6]:
print(prices.columns)
print(load.columns)

Index(['timestamp', 'price', 'price_count', 'price_valid'], dtype='str')
Index(['timestamp', 'load_forecast_mw', 'load_count', 'load_valid'], dtype='str')


In [7]:
master = (
    prices
    .merge(weather, on="timestamp", how="inner")
    .merge(load, on="timestamp", how="inner")
)

In [8]:
master.shape

(18816, 13)

In [9]:
master.head()

,timestamp,price,price_count,price_valid,temp_stockholm,wind_stockholm,temp_gothenburg,wind_gothenburg,temp_orebro,wind_orebro,load_forecast_mw,load_count,load_valid
0,2024-03-08 00:00:00+00:00,64.89,1,True,-1.0,5.15,-0.8,3.99,-1.5,5.64,10612.0,1,True
1,2024-03-08 01:00:00+00:00,64.87,1,True,-1.3,5.21,-1.0,3.79,-1.8,5.64,10566.0,1,True
2,2024-03-08 02:00:00+00:00,64.01,1,True,-1.7,5.50,-1.0,3.49,-2.1,5.32,10621.0,1,True
3,2024-03-08 03:00:00+00:00,64.24,1,True,-1.9,5.71,-1.1,3.45,-2.4,5.10,10737.0,1,True
4,2024-03-08 04:00:00+00:00,65.03,1,True,-2.1,5.92,-1.4,3.86,-2.8,5.05,11201.0,1,True


In [10]:
master.isna().sum()

timestamp             0
price               459
price_count           0
price_valid           0
temp_stockholm        0
wind_stockholm        0
temp_gothenburg       0
wind_gothenburg       0
temp_orebro           0
wind_orebro           0
load_forecast_mw     24
load_count            0
load_valid            0
dtype: int64

In [11]:
master["price_24h"] = master["price"].shift(24) #priset 24h innan
master["price_48h"] = master["price"].shift(48)
master["price_168h"] = master["price"].shift(168) #priset 1 vecka innan


In [12]:
master [
    ["timestamp", "price", "price_24h", "price_48h", "price_168h"]
].head(170)

,timestamp,price,price_24h,price_48h,price_168h
0,2024-03-08 00:00:00+00:00,64.89,NaN,NaN,NaN
1,2024-03-08 01:00:00+00:00,64.87,NaN,NaN,NaN
2,2024-03-08 02:00:00+00:00,64.01,NaN,NaN,NaN
3,2024-03-08 03:00:00+00:00,64.24,NaN,NaN,NaN
4,2024-03-08 04:00:00+00:00,65.03,NaN,NaN,NaN
...,...,...,...,...,...
165,2024-03-14 21:00:00+00:00,0.10,25.91,68.27,NaN
166,2024-03-14 22:00:00+00:00,-0.09,12.50,63.15,NaN
167,2024-03-14 23:00:00+00:00,-0.11,26.21,58.86,NaN
168,2024-03-15 00:00:00+00:00,NaN,20.98,57.34,64.89


In [13]:
master["local_time"] = master["timestamp"].dt.tz_convert("Europe/Stockholm")

In [14]:
master[["timestamp", "local_time"]].head()

,timestamp,local_time
0,2024-03-08 00:00:00+00:00,2024-03-08 01:00:00+01:00
1,2024-03-08 01:00:00+00:00,2024-03-08 02:00:00+01:00
2,2024-03-08 02:00:00+00:00,2024-03-08 03:00:00+01:00
3,2024-03-08 03:00:00+00:00,2024-03-08 04:00:00+01:00
4,2024-03-08 04:00:00+00:00,2024-03-08 05:00:00+01:00


In [15]:
master["hour"] = master["local_time"].dt.hour
master["weekday"] = master["local_time"].dt.dayofweek
master["month"] = master["local_time"].dt.month
master["is_weekend"] = master["weekday"] >= 5

In [16]:
master[
    ["local_time", "hour", "weekday", "month", "is_weekend"]
].head(40)

,local_time,hour,weekday,month,is_weekend
0,2024-03-08 01:00:00+01:00,1,4,3,False
1,2024-03-08 02:00:00+01:00,2,4,3,False
2,2024-03-08 03:00:00+01:00,3,4,3,False
3,2024-03-08 04:00:00+01:00,4,4,3,False
4,2024-03-08 05:00:00+01:00,5,4,3,False
5,2024-03-08 06:00:00+01:00,6,4,3,False
6,2024-03-08 07:00:00+01:00,7,4,3,False
7,2024-03-08 08:00:00+01:00,8,4,3,False
8,2024-03-08 09:00:00+01:00,9,4,3,False
9,2024-03-08 10:00:00+01:00,10,4,3,False


In [17]:
master["split"] = "train"
master.loc[
    master["timestamp"] >= pd.Timestamp("2025-03-08 00:00", tz="UTC"), 
    "split"
] = "validation"

master.loc[
    master["timestamp"] >= pd.Timestamp("2025-05-01 00:00", tz="UTC"), 
    "split"
] = "test"

In [18]:
master["split"].value_counts()

split
train         8760
test          8760
validation    1296
Name: count, dtype: int64

In [19]:
master.groupby("split")["timestamp"].agg(["min", "max"])

,min,max
split,,
test,2025-05-01 00:00:00+00:00,2026-04-30 23:00:00+00:00
train,2024-03-08 00:00:00+00:00,2025-03-07 23:00:00+00:00
validation,2025-03-08 00:00:00+00:00,2025-04-30 23:00:00+00:00


In [20]:
master["timestamp"].duplicated().sum()

np.int64(0)

In [21]:
master["timestamp"].diff().value_counts().head()

timestamp
0 days 01:00:00    18815
Name: count, dtype: int64

In [22]:
master.to_csv(
    "../data/processed/se3_master_dataset.csv", 
    index=False
)